# DeepSpeed: Training Massive Models Without Running Out of Memory

## What Is DeepSpeed?

Imagine you are trying to move a grand piano (a huge AI model) but your truck (GPU) can only carry half of it.  
DeepSpeed is like hiring a moving crew that splits the piano across multiple trucks — each truck carries a piece, and together they move the whole thing.

**DeepSpeed** is an open-source deep learning optimization library from Microsoft.  
It lets you train models that are too large to fit in a single GPU by:
- Splitting the model across many GPUs (ZeRO)
- Using 16-bit instead of 32-bit numbers (mixed precision)
- Offloading data to cheaper memory like CPU RAM or NVMe storage

## Why Does This Matter?

GPT-3 has 175 billion parameters. Storing them in full precision (fp32) takes **700 GB of GPU memory**.  
A top-of-the-line A100 GPU has only 80 GB. You would need 9 GPUs just to hold the weights — and that is before any training memory!

DeepSpeed makes it possible to train such models on realistic hardware budgets.

## Official Resources

- **Documentation**: [https://deepspeed.readthedocs.io/](https://deepspeed.readthedocs.io/)
- **GitHub**: [https://github.com/microsoft/DeepSpeed](https://github.com/microsoft/DeepSpeed)
- **ZeRO Paper**: [https://arxiv.org/abs/1910.02054](https://arxiv.org/abs/1910.02054)
- **ZeRO-Offload Paper**: [https://arxiv.org/abs/2101.06840](https://arxiv.org/abs/2101.06840)
- **YouTube — DeepSpeed Overview (Microsoft Research)**: [https://www.youtube.com/watch?v=wbG2ZEDPIyw](https://www.youtube.com/watch?v=wbG2ZEDPIyw)
- **YouTube — ZeRO Explained Simply**: [https://www.youtube.com/watch?v=y4_bCiAsIAk](https://www.youtube.com/watch?v=y4_bCiAsIAk)

## Prerequisites

Before reading this notebook you should know:
- Basic PyTorch: how to define a model, loss function, and optimizer
- What a GPU is and why ML training uses them
- Basic idea of distributed training (running code on multiple machines/GPUs)

You do **not** need a GPU to follow this notebook.  
All GPU-dependent code is guarded with availability checks and shows realistic simulated output.

## Installation

```bash
# Basic install (CPU only or CUDA auto-detected)
pip install deepspeed

# Verify installation
ds_report
```

DeepSpeed requires a Linux system with CUDA for full GPU functionality.  
On macOS/Windows it installs but some features are CPU-only.

In [ ]:
# Check availability
try:
    import deepspeed
    DEEPSPEED_AVAILABLE = True
    print(f"DeepSpeed version: {deepspeed.__version__}")
except ImportError:
    DEEPSPEED_AVAILABLE = False
    print("DeepSpeed not installed — simulated outputs will be shown throughout.")
    print("Install with: pip install deepspeed")

import torch
import torch.nn as nn
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Core Concept 1: Why Training Is a Memory Problem

When you train a model, the GPU must hold several things at once:

| What | Memory Cost (per parameter) |
|------|-----------------------------|
| Model weights (fp32) | 4 bytes |
| Gradients | 4 bytes |
| Optimizer states (Adam: momentum + variance) | 8 bytes |
| **Total per parameter** | **16 bytes** |

For a 7-billion parameter model (like Llama-2-7B):
- Weights alone: 7B × 4 bytes = **28 GB**
- With optimizer states: 7B × 16 bytes = **112 GB**

A single A100 has 80 GB. You are already over budget **before** factoring in activations.

This is the problem DeepSpeed solves.

In [ ]:
def estimate_training_memory(num_params_billions, precision='fp32', zero_stage=0, num_gpus=1):
    """
    Estimate GPU memory required for training a model.
    
    Args:
        num_params_billions: Model size in billions of parameters
        precision: 'fp32' or 'fp16'/'bf16'
        zero_stage: ZeRO optimization stage (0, 1, 2, or 3)
        num_gpus: Number of GPUs (only matters for ZeRO stage 1/2/3)
    """
    P = num_params_billions * 1e9  # total parameters
    
    # Bytes per element
    weight_bytes = 2 if precision in ('fp16', 'bf16') else 4
    grad_bytes   = 2 if precision in ('fp16', 'bf16') else 4
    # Adam optimizer states are always fp32 internally
    opt_bytes    = 8  # momentum (4) + variance (4)
    
    weights    = P * weight_bytes
    grads      = P * grad_bytes
    opt_states = P * opt_bytes
    
    total_no_zero = weights + grads + opt_states
    
    # ZeRO partitions across GPUs
    if zero_stage == 0:
        per_gpu = total_no_zero
        saved = 0
    elif zero_stage == 1:
        # Optimizer states partitioned
        per_gpu = weights + grads + opt_states / num_gpus
        saved = opt_states * (1 - 1/num_gpus)
    elif zero_stage == 2:
        # Optimizer states + gradients partitioned
        per_gpu = weights + (grads + opt_states) / num_gpus
        saved = (grads + opt_states) * (1 - 1/num_gpus)
    elif zero_stage == 3:
        # Everything partitioned
        per_gpu = (weights + grads + opt_states) / num_gpus
        saved = total_no_zero * (1 - 1/num_gpus)
    
    def gb(b): return b / 1e9
    
    print(f"Model: {num_params_billions}B parameters | Precision: {precision} | ZeRO stage {zero_stage} | {num_gpus} GPU(s)")
    print(f"-" * 70)
    print(f"  Weights:       {gb(weights):8.1f} GB")
    print(f"  Gradients:     {gb(grads):8.1f} GB")
    print(f"  Optimizer:     {gb(opt_states):8.1f} GB")
    print(f"  Total (naive): {gb(total_no_zero):8.1f} GB")
    print(f"  Per-GPU (ZeRO-{zero_stage}): {gb(per_gpu):8.1f} GB  (saved {gb(saved):.1f} GB)")
    print()
    return gb(per_gpu)


# Compare ZeRO stages for a 7B model on 8 GPUs
print("=" * 70)
print("Memory estimates for a 7B parameter model (e.g., Llama-2-7B)")
print("=" * 70)
for stage in [0, 1, 2, 3]:
    estimate_training_memory(7, precision='fp16', zero_stage=stage, num_gpus=8)

## Core Concept 2: ZeRO — Zero Redundancy Optimizer

ZeRO is DeepSpeed's core innovation. The insight: in standard distributed training, **every GPU stores a full copy** of weights, gradients, and optimizer states — that is pure redundancy.

ZeRO eliminates this redundancy in three stages:

### Analogy: Splitting a Restaurant Bill

Imagine 4 friends at a restaurant. The naive approach: each person carries the full bill in their wallet (4× redundancy).  
ZeRO says: split the bill!

| Stage | What Gets Split | Analogy |
|-------|-----------------|--------|
| ZeRO-1 | Optimizer states | Each friend pays 1/4 of the tip |
| ZeRO-2 | + Gradients | Each friend also tracks 1/4 of the receipt items |
| ZeRO-3 | + Model weights | Each friend even memorizes 1/4 of the menu |

### Memory Savings (8 GPUs)

| Stage | Memory Reduction |
|-------|------------------|
| ZeRO-1 | ~4× (optimizer states dominate) |
| ZeRO-2 | ~8× |
| ZeRO-3 | ~64× (full model sharding) |

### Communication Trade-off

Splitting data means GPUs need to communicate more.  
ZeRO-3 has higher communication overhead than ZeRO-1, but the memory savings are often worth it.

In [ ]:
# Visualize ZeRO stages conceptually

stages = {
    'No ZeRO\n(DDP)': {
        'gpu0': ['W', 'G', 'O'],
        'gpu1': ['W', 'G', 'O'],
        'gpu2': ['W', 'G', 'O'],
        'gpu3': ['W', 'G', 'O'],
    },
    'ZeRO-1\n(opt shard)': {
        'gpu0': ['W', 'G', 'O[0]'],
        'gpu1': ['W', 'G', 'O[1]'],
        'gpu2': ['W', 'G', 'O[2]'],
        'gpu3': ['W', 'G', 'O[3]'],
    },
    'ZeRO-2\n(+grad shard)': {
        'gpu0': ['W', 'G[0]', 'O[0]'],
        'gpu1': ['W', 'G[1]', 'O[1]'],
        'gpu2': ['W', 'G[2]', 'O[2]'],
        'gpu3': ['W', 'G[3]', 'O[3]'],
    },
    'ZeRO-3\n(+weight shard)': {
        'gpu0': ['W[0]', 'G[0]', 'O[0]'],
        'gpu1': ['W[1]', 'G[1]', 'O[1]'],
        'gpu2': ['W[2]', 'G[2]', 'O[2]'],
        'gpu3': ['W[3]', 'G[3]', 'O[3]'],
    },
}

print("ZeRO Sharding Across 4 GPUs")
print("Legend: W=Weights, G=Gradients, O=Optimizer states, [n]=shard n\n")

for stage_name, gpus in stages.items():
    print(f"  {stage_name}")
    for gpu, contents in gpus.items():
        print(f"    {gpu}: {' | '.join(contents)}")
    print()

## Core Concept 3: Mixed Precision Training

### The Idea

Normal training uses 32-bit floats (fp32) for everything.  
Mixed precision uses 16-bit floats (fp16 or bf16) for most computation, keeping fp32 only where precision matters.

Benefits:
- 2× less memory for weights and activations
- 2–8× faster matrix multiplications on modern GPUs (Tensor Cores)

### fp16 vs bf16

| Property | fp16 | bf16 |
|----------|------|------|
| Bits | 16 | 16 |
| Range | ±65504 | ±3.4×10³⁸ (same as fp32!) |
| Precision | Higher | Lower |
| Risk | Overflow/underflow | Almost none |
| Loss scaling needed? | Yes | No |
| GPU support | V100+ | Ampere+ (A100, RTX 3090) |

**Rule of thumb**: Use bf16 if your GPU supports it (A100, H100). Use fp16 otherwise.

### Loss Scaling (fp16 only)

Gradients in fp16 can underflow to zero (they are very small numbers).  
Loss scaling multiplies the loss by a large number before backward pass,  
then divides the gradients back — keeping them in the representable fp16 range.

In [ ]:
# Demonstrate why fp16 overflows and bf16 doesn't
import struct

fp32_max = 3.4e38
fp16_max = 65504.0
# bf16 shares the same exponent bits as fp32, so same range
bf16_max = fp32_max  # approximately

test_value = 100_000.0  # A large activation value

print(f"Test value: {test_value:,.0f}")
print(f"fp32 max:  {fp32_max:.2e}  -> Can represent {test_value}? {test_value <= fp32_max}")
print(f"fp16 max:  {fp16_max:.2e}  -> Can represent {test_value}? {test_value <= fp16_max} (OVERFLOW!)")
print(f"bf16 max:  {bf16_max:.2e}  -> Can represent {test_value}? {test_value <= bf16_max}")
print()

# Demonstrate loss scaling concept
import torch

print("Loss Scaling Demo:")
tiny_grad = torch.tensor(1e-8)  # This would underflow to 0 in fp16
scale = 1024.0
scaled_grad = tiny_grad * scale
unscaled_grad = scaled_grad / scale

print(f"  Original gradient: {tiny_grad.item():.2e}")
print(f"  After scaling by {scale}: {scaled_grad.item():.2e}  (stays non-zero)")
print(f"  After unscaling:  {unscaled_grad.item():.2e}  (correct value recovered)")
print()
print("Automatic loss scaling adjusts scale dynamically — grows when no overflow, shrinks on overflow.")

## Core Concept 4: Gradient Checkpointing

During the forward pass, PyTorch saves all intermediate activations so it can compute gradients during the backward pass.  
For a deep model, these activations consume a lot of memory.

**Gradient checkpointing** trades compute for memory:  
- Save only some activations during forward pass (the checkpoints)  
- Recompute the rest during backward pass when needed

**Analogy**: Instead of printing and saving every page of a 1000-page book while reading, you save only every 100th page and reread sections you need.

**Memory cost**: Reduces activation memory from O(L) to O(√L) where L is number of layers.  
**Compute cost**: ~33% more compute (each activation recomputed once).

In [ ]:
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

class TransformerBlock(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads=4, batch_first=True)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ff(x))
        return x


class ModelWithCheckpointing(nn.Module):
    def __init__(self, num_layers=6, d_model=256, use_checkpointing=False):
        super().__init__()
        self.layers = nn.ModuleList([TransformerBlock(d_model) for _ in range(num_layers)])
        self.use_checkpointing = use_checkpointing

    def forward(self, x):
        for layer in self.layers:
            if self.use_checkpointing:
                # Recompute activations during backward — saves memory
                x = checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        return x


# Compare memory usage
batch, seq_len, d_model = 4, 128, 256
x = torch.randn(batch, seq_len, d_model)

for use_ckpt in [False, True]:
    model = ModelWithCheckpointing(num_layers=6, use_checkpointing=use_ckpt)
    
    if torch.cuda.is_available():
        model, x_gpu = model.cuda(), x.cuda()
        torch.cuda.reset_peak_memory_stats()
        out = model(x_gpu)
        out.sum().backward()
        peak_mb = torch.cuda.max_memory_allocated() / 1e6
        label = "checkpointing" if use_ckpt else "standard"
        print(f"{label:15s}: peak GPU memory = {peak_mb:.1f} MB")
    else:
        out = model(x)
        label = "checkpointing" if use_ckpt else "standard  "
        print(f"{label}: forward pass OK (no GPU — memory comparison skipped)")

print("\nGradient checkpointing reduces activation memory at the cost of ~33% more compute.")

## DeepSpeed Configuration (JSON)

DeepSpeed is configured via a JSON file or Python dictionary.  
Here are the most important fields:

In [ ]:
import json

# ── Config for ZeRO-2 with fp16 and CPU offload ──────────────────────────────
ds_config_zero2 = {
    # Total batch size across all GPUs and gradient accumulation steps
    "train_batch_size": 32,
    "train_micro_batch_size_per_gpu": 4,   # per-GPU batch
    "gradient_accumulation_steps": 4,      # 4 GPUs × 4 micro × 4 accum = 64 total

    # Mixed precision — use fp16 on older GPUs (V100), bf16 on A100+
    "fp16": {
        "enabled": True,
        "loss_scale": 0,              # 0 = dynamic loss scaling
        "loss_scale_window": 1000,    # adjust scale every 1000 steps
        "hysteresis": 2,
        "min_loss_scale": 1
    },

    # ZeRO optimization
    "zero_optimization": {
        "stage": 2,                   # 0=off, 1=opt, 2=opt+grad, 3=opt+grad+weights
        "allgather_partitions": True,
        "allgather_bucket_size": 2e8, # 200 MB buckets for communication
        "reduce_scatter": True,
        "reduce_bucket_size": 2e8,
        "overlap_comm": True,         # overlap compute and communication
        "contiguous_gradients": True, # extra memory savings
        # Offload optimizer states to CPU (saves GPU memory at cost of speed)
        "offload_optimizer": {
            "device": "cpu",
            "pin_memory": True        # pinned memory = faster CPU↔GPU transfer
        }
    },

    # Gradient clipping to prevent exploding gradients
    "gradient_clipping": 1.0,

    # Logging
    "steps_per_print": 100,
    "wall_clock_breakdown": False
}

print("ZeRO-2 Config:")
print(json.dumps(ds_config_zero2, indent=2))

# Save to file (DeepSpeed expects a JSON file path or dict)
with open("/tmp/ds_config_zero2.json", "w") as f:
    json.dump(ds_config_zero2, f, indent=2)
print("\nConfig saved to /tmp/ds_config_zero2.json")

In [ ]:
# ── ZeRO-3 config with both CPU offload (for extreme memory savings) ─────────
ds_config_zero3 = {
    "train_batch_size": 16,
    "train_micro_batch_size_per_gpu": 1,
    "gradient_accumulation_steps": 16,

    # Use bf16 on A100+ GPUs (no loss scaling needed)
    "bf16": {
        "enabled": True
    },

    "zero_optimization": {
        "stage": 3,
        # Gather parameters just before use, then free them
        "stage3_prefetch_bucket_size": 5e7,
        "stage3_param_persistence_threshold": 1e5,
        "stage3_max_live_parameters": 1e9,
        "stage3_max_reuse_distance": 1e9,

        # Offload model parameters to CPU (ZeRO-Offload)
        "offload_param": {
            "device": "cpu",
            "pin_memory": True
        },
        # Offload optimizer to CPU too
        "offload_optimizer": {
            "device": "cpu",
            "pin_memory": True
        }
    },

    "gradient_clipping": 1.0
}

print("ZeRO-3 + CPU Offload Config:")
print(json.dumps(ds_config_zero3, indent=2))

## The `deepspeed.initialize()` API

This is the single entry point that wraps your PyTorch model, optimizer, and config.

```python
model_engine, optimizer, dataloader, lr_scheduler = deepspeed.initialize(
    model=model,           # your nn.Module
    optimizer=optimizer,   # your torch.optim.Optimizer (or None to use DS built-in)
    model_parameters=model.parameters(),  # optional if passing optimizer
    training_data=dataset, # optional DataLoader setup
    lr_scheduler=scheduler,
    config=ds_config,      # dict or path to JSON file
)
```

The returned `model_engine` replaces your model.  
It handles backward, optimizer step, and gradient scaling automatically.

In [ ]:
import torch
import torch.nn as nn

# ── Define a simple GPT-2-style language model ───────────────────────────────
class MiniGPT(nn.Module):
    def __init__(self, vocab_size=50257, d_model=768, num_layers=12, num_heads=12, seq_len=1024):
        super().__init__()
        self.embed    = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(seq_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=d_model*4,
            dropout=0.1, batch_first=True, norm_first=True  # Pre-LN (GPT-style)
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.lm_head  = nn.Linear(d_model, vocab_size, bias=False)
        # Weight tying: embed and lm_head share weights (saves memory)
        self.lm_head.weight = self.embed.weight

    def forward(self, input_ids):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0)  # (1, T)
        x = self.embed(input_ids) + self.pos_embed(positions)
        x = self.transformer(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)
        return logits


# Count parameters
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

# Small model for demo
model = MiniGPT(vocab_size=32000, d_model=512, num_layers=6, num_heads=8, seq_len=512)
total, trainable = count_params(model)
print(f"MiniGPT parameters: {total/1e6:.1f}M total, {trainable/1e6:.1f}M trainable")
print(f"Memory (fp32):  {total * 4 / 1e6:.0f} MB weights alone")
print(f"Memory (fp16):  {total * 2 / 1e6:.0f} MB weights alone")
print(f"Training total (fp32, no ZeRO): {total * 16 / 1e6:.0f} MB")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# ── Synthetic token dataset ───────────────────────────────────────────────────
class SyntheticTokenDataset(Dataset):
    def __init__(self, num_samples=1000, seq_len=128, vocab_size=32000):
        self.data = torch.randint(0, vocab_size, (num_samples, seq_len))

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]


dataset = SyntheticTokenDataset()
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)
print(f"Dataset: {len(dataset)} sequences of length 128")
print(f"Batches per epoch: {len(dataloader)}")


# ── Training loop: standard PyTorch vs DeepSpeed ─────────────────────────────

if DEEPSPEED_AVAILABLE:
    import deepspeed

    ds_config = {
        "train_batch_size": 8,
        "train_micro_batch_size_per_gpu": 8,
        "fp16": {"enabled": torch.cuda.is_available()},
        "zero_optimization": {"stage": 2},
        "gradient_clipping": 1.0,
        "steps_per_print": 10,
    }

    # Initialize: model + optimizer wrapped by DeepSpeed
    model_engine, optimizer, _, _ = deepspeed.initialize(
        model=model,
        model_parameters=model.parameters(),
        config=ds_config,
    )

    print("DeepSpeed training loop:")
    loss_fn = nn.CrossEntropyLoss()

    for step, batch in enumerate(dataloader):
        if step >= 3: break  # Demo: just 3 steps

        batch = batch.to(model_engine.device)
        inputs, targets = batch[:, :-1], batch[:, 1:]

        logits = model_engine(inputs)          # forward
        loss   = loss_fn(logits.reshape(-1, 32000), targets.reshape(-1))

        model_engine.backward(loss)            # handles fp16 loss scaling automatically
        model_engine.step()                    # optimizer step + gradient sync

        print(f"  Step {step+1}: loss = {loss.item():.4f}")

else:
    # ── Simulated DeepSpeed output ────────────────────────────────────────────
    print("DeepSpeed not installed — showing simulated training output:")
    print()
    print("  [DeepSpeed] Using ZeRO stage 2 with fp16 mixed precision")
    print("  [DeepSpeed] Optimizer: Adam, Learning rate: 1e-4")
    print("  [DeepSpeed] Gradient accumulation steps: 1")
    print("  [DeepSpeed] Total parameters: 85.7M")
    print()
    import math
    for step in range(1, 4):
        fake_loss = 10.8 - step * 0.15 + 0.05 * (step % 2)  # realistic LM loss
        print(f"  Step {step}: loss = {fake_loss:.4f}")

    print()
    print("  Key difference from standard PyTorch training:")
    print("    Standard: loss.backward() -> optimizer.step()")
    print("    DeepSpeed: model_engine.backward(loss) -> model_engine.step()")
    print("    (DeepSpeed engine handles loss scaling, gradient sync, ZeRO communication)")

## ZeRO-Offload and ZeRO-Infinity

When even multiple GPUs aren't enough, DeepSpeed can offload data to cheaper memory:

| Technique | Where Data Goes | Memory Available | Speed Impact |
|-----------|-----------------|------------------|--------------|
| GPU only  | GPU HBM | 40–80 GB per GPU | Fastest |
| ZeRO-Offload | CPU RAM | 256 GB+ typical | ~10% slower |
| ZeRO-Infinity | NVMe SSD | Terabytes | Much slower |

### When to Use Each

- **GPU only**: You have enough GPUs. Always prefer this.
- **ZeRO-Offload**: Training a model that barely doesn't fit across your GPUs. CPU RAM is the overflow valve.
- **ZeRO-Infinity**: Research scenarios where you want to experiment with trillion-parameter models on consumer hardware. Slow but possible.

### The NVMe Offload Paper

[ZeRO-Infinity Paper](https://arxiv.org/abs/2104.07857) — trained a 32-trillion parameter model on 512 V100 GPUs!

In [ ]:
# ZeRO-Offload config (CPU RAM offload)
import json

ds_config_offload = {
    "train_batch_size": 8,
    "fp16": {"enabled": True},
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {
            "device": "cpu",       # Send optimizer states to CPU
            "pin_memory": True     # Pinned memory = faster PCIe transfers
        },
        "offload_param": {         # ZeRO-3 only: also offload parameters
            "device": "cpu",
            "pin_memory": True
        }
    }
}

# ZeRO-Infinity config (NVMe offload)
ds_config_infinity = {
    "train_batch_size": 4,
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {
            "device": "nvme",                  # NVMe SSD!
            "nvme_path": "/local_nvme",        # Path to NVMe mount
            "pin_memory": True,
            "buffer_count": 4,
            "fast_init": False
        },
        "offload_param": {
            "device": "nvme",
            "nvme_path": "/local_nvme",
            "pin_memory": True,
            "buffer_count": 5,
            "buffer_size": 1e8,
            "max_in_cpu": 1e9
        },
        "aio": {                               # Async I/O settings
            "block_size": 1048576,
            "queue_depth": 8,
            "thread_count": 1,
            "single_submit": False,
            "overlap_events": True
        }
    }
}

print("ZeRO-Offload config (CPU RAM):")
print(json.dumps(ds_config_offload, indent=2))
print("\nZeRO-Infinity config (NVMe SSD):")
print(json.dumps(ds_config_infinity, indent=2))

## Launching DeepSpeed Training

DeepSpeed uses its own launcher instead of `torchrun` or `python -m torch.distributed.launch`.

```bash
# Single node, all available GPUs
deepspeed train.py --deepspeed_config ds_config.json

# Specify number of GPUs
deepspeed --num_gpus=4 train.py --deepspeed_config ds_config.json

# Multi-node (2 nodes, 8 GPUs each = 16 GPUs total)
deepspeed --hostfile=hostfile --num_nodes=2 --num_gpus=8 train.py

# With Hugging Face Trainer (built-in DeepSpeed support)
python -m torch.distributed.run --nproc_per_node=4 train.py \
    --deepspeed ds_config.json
```

### Hostfile format for multi-node:
```
worker-1 slots=8
worker-2 slots=8
```

### Integration with Hugging Face Transformers

Hugging Face `Trainer` has first-class DeepSpeed support — just pass `--deepspeed ds_config.json`  
to your training script. No code changes needed!

In [ ]:
# Example: Hugging Face Trainer with DeepSpeed
# This shows how little code change is needed

HF_AVAILABLE = False
try:
    from transformers import TrainingArguments, Trainer, AutoModelForCausalLM, AutoTokenizer
    HF_AVAILABLE = True
except ImportError:
    pass

if HF_AVAILABLE and DEEPSPEED_AVAILABLE:
    training_args = TrainingArguments(
        output_dir="./output",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        fp16=True,
        deepspeed="/tmp/ds_config_zero2.json",  # Just add this line!
        logging_steps=10,
        save_steps=500,
    )
    print("TrainingArguments configured with DeepSpeed ZeRO-2")
else:
    print("Hugging Face Trainer + DeepSpeed integration (simulated):")
    print()
    print("  from transformers import TrainingArguments, Trainer")
    print("")
    print("  training_args = TrainingArguments(")
    print("      output_dir='./output',")
    print("      per_device_train_batch_size=4,")
    print("      num_train_epochs=3,")
    print("      fp16=True,")
    print("      deepspeed='ds_config.json',  # <-- only change needed!")
    print("  )")
    print()
    print("  trainer = Trainer(model=model, args=training_args, ...)")
    print("  trainer.train()  # DeepSpeed handles everything under the hood")

## DeepSpeed vs PyTorch FSDP

FSDP (Fully Sharded Data Parallel) is PyTorch's native answer to ZeRO-3.  
Both do full model sharding — here is how they compare:

| Feature | DeepSpeed ZeRO | PyTorch FSDP |
|---------|---------------|-------------|
| Origin | Microsoft | Meta / PyTorch |
| Requires extra install | Yes (`pip install deepspeed`) | No (built into PyTorch) |
| Config style | JSON file | Python API |
| Stages | ZeRO 1/2/3 | Mixed precision sharding |
| CPU/NVMe offload | Yes (ZeRO-Infinity) | Limited |
| Hugging Face support | Native | Native |
| Debugging ease | Harder (less Pythonic) | Easier (pure PyTorch) |
| Performance | Slightly better at scale | Simpler to tune |
| Best for | Very large models, multi-node | PyTorch-native codebases |

**Rule of thumb**:
- Model fits on <32 GPUs with FSDP → use FSDP (simpler)
- Need CPU/NVMe offload or training 100B+ models → use DeepSpeed

In [ ]:
# FSDP example for comparison (pure PyTorch, no DeepSpeed)
import torch
import torch.nn as nn

FSDP_AVAILABLE = False
try:
    from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
    from torch.distributed.fsdp import MixedPrecision
    from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy
    import functools
    FSDP_AVAILABLE = True
except ImportError:
    pass

if FSDP_AVAILABLE and torch.distributed.is_available():
    # In a real script, torch.distributed.init_process_group() would be called first
    fp16_policy = MixedPrecision(
        param_dtype=torch.float16,
        reduce_dtype=torch.float16,
        buffer_dtype=torch.float16,
    )
    wrap_policy = functools.partial(size_based_auto_wrap_policy, min_num_params=1e6)
    # fsdp_model = FSDP(model, mixed_precision=fp16_policy, auto_wrap_policy=wrap_policy)
    print("FSDP config objects created (not initialized without distributed process group)")
else:
    print("PyTorch FSDP vs DeepSpeed ZeRO (conceptual comparison):")
    print()
    print("  # DeepSpeed approach: JSON config + deepspeed.initialize()")
    print("  model_engine, optimizer, _, _ = deepspeed.initialize(")
    print("      model=model, config='ds_config.json'")
    print("  )")
    print()
    print("  # FSDP approach: Python API wrapping")
    print("  from torch.distributed.fsdp import FullyShardedDataParallel as FSDP")
    print("  model = FSDP(model, mixed_precision=fp16_policy, auto_wrap_policy=wrap_policy)")
    print()
    print("  Both achieve the same goal: sharding model across GPUs.")
    print("  FSDP = simpler, native. DeepSpeed = more features, more config.")

## Common Pitfalls and How to Avoid Them

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Wrong `train_batch_size` | AssertionError on startup | Must equal `micro_batch × grad_accum × num_gpus` |
| fp16 overflow | `loss = nan`, `loss_scale` → 1 | Switch to bf16, or add `hysteresis` in fp16 config |
| ZeRO-3 with model.parameters() | Parameters not gathered | Use `model_engine.parameters()` not `model.parameters()` after init |
| Checkpoint loading with ZeRO | Mismatch errors | Use `deepspeed.load_checkpoint()` not `torch.load()` |
| Slow training with ZeRO-3 | Lower GPU utilization | Increase `stage3_prefetch_bucket_size`, or try ZeRO-2 |
| Out of CPU memory with offload | System RAM OOM | Reduce `offload_param` buffer sizes |
| Inference with ZeRO-3 | Parameters not gathered | Use `model_engine.module` after `gather_16bit_weights_on_model_save` |

In [ ]:
# Common mistake: wrong batch size math
def validate_deepspeed_batch_config(train_batch_size, micro_batch_per_gpu, grad_accum_steps, num_gpus):
    expected = micro_batch_per_gpu * grad_accum_steps * num_gpus
    ok = (train_batch_size == expected)
    print(f"train_batch_size: {train_batch_size}")
    print(f"micro_batch_per_gpu ({micro_batch_per_gpu}) × grad_accum ({grad_accum_steps}) × num_gpus ({num_gpus}) = {expected}")
    if ok:
        print("PASS: batch size configuration is consistent")
    else:
        print(f"FAIL: DeepSpeed will raise AssertionError at startup!")
        print(f"      Fix: set train_batch_size={expected} or adjust the other values.")
    return ok

print("Correct config:")
validate_deepspeed_batch_config(train_batch_size=64, micro_batch_per_gpu=4,
                                grad_accum_steps=4, num_gpus=4)
print()
print("Incorrect config:")
validate_deepspeed_batch_config(train_batch_size=32, micro_batch_per_gpu=4,
                                grad_accum_steps=4, num_gpus=4)

## Mini Project: Efficient Training Configuration Wizard

**Goal**: Given a model size and available hardware, generate the optimal DeepSpeed config.

This simulates the thought process of an ML engineer planning a large-scale training run.

In [ ]:
import json
import math

def deepspeed_config_wizard(
    model_params_billions: float,
    gpu_memory_gb: float,
    num_gpus: int,
    cpu_ram_gb: float = 64,
    gpu_generation: str = 'ampere',  # 'ampere' (A100), 'volta' (V100)
    target_batch_size: int = 32,
):
    """
    Generate a recommended DeepSpeed config for given hardware.
    
    Returns recommended config dict and explanation.
    """
    P = model_params_billions
    total_gpu_gb = gpu_memory_gb * num_gpus
    
    # Memory estimates (fp16/bf16 training)
    weights_gb  = P * 2  # fp16 weights
    grads_gb    = P * 2  # fp16 gradients
    opt_gb      = P * 8  # fp32 optimizer states
    act_gb      = P * 0.5  # rough activation estimate
    total_gb    = weights_gb + grads_gb + opt_gb + act_gb
    
    print(f"=" * 65)
    print(f"DeepSpeed Config Wizard")
    print(f"=" * 65)
    print(f"Model:       {P}B parameters")
    print(f"Hardware:    {num_gpus}× {gpu_memory_gb}GB GPU = {total_gpu_gb}GB total")
    print(f"CPU RAM:     {cpu_ram_gb}GB")
    print(f"GPU gen:     {gpu_generation}")
    print(f"")
    print(f"Memory required (fp16, no ZeRO): {total_gb:.0f} GB")
    print(f"Memory per GPU (ZeRO-3):         {total_gb/num_gpus:.1f} GB")
    print()
    
    # Determine ZeRO stage needed
    if total_gb <= total_gpu_gb * 0.8:  # 80% utilization
        zero_stage = 1
        reason = "ZeRO-1 sufficient — model fits across GPUs"
    elif (weights_gb + (grads_gb + opt_gb) / num_gpus) <= gpu_memory_gb * 0.8:
        zero_stage = 2
        reason = "ZeRO-2 needed — optimizer + gradient sharding required"
    elif total_gb / num_gpus <= gpu_memory_gb * 0.8:
        zero_stage = 3
        reason = "ZeRO-3 needed — full model sharding required"
    elif total_gb / num_gpus <= (gpu_memory_gb + cpu_ram_gb / num_gpus):
        zero_stage = 3
        reason = "ZeRO-3 + CPU offload needed — model too large even with ZeRO-3"
    else:
        zero_stage = 3
        reason = "ZeRO-3 + NVMe offload needed — extreme scale"
    
    use_offload = total_gb / num_gpus > gpu_memory_gb * 0.8
    use_bf16 = gpu_generation.lower() in ('ampere', 'hopper', 'ada')
    
    # Compute batch size config
    micro_batch = max(1, min(8, int(gpu_memory_gb // (P * 2 * 0.1))))
    grad_accum = max(1, target_batch_size // (micro_batch * num_gpus))
    actual_batch = micro_batch * grad_accum * num_gpus
    
    print(f"Recommendation: {reason}")
    print(f"ZeRO stage: {zero_stage}")
    print(f"Precision:  {'bf16' if use_bf16 else 'fp16'}")
    print(f"CPU offload: {use_offload}")
    print()
    
    # Build config
    config = {
        "train_batch_size": actual_batch,
        "train_micro_batch_size_per_gpu": micro_batch,
        "gradient_accumulation_steps": grad_accum,
        "gradient_clipping": 1.0,
        "steps_per_print": 50,
    }
    
    if use_bf16:
        config["bf16"] = {"enabled": True}
    else:
        config["fp16"] = {"enabled": True, "loss_scale": 0, "loss_scale_window": 1000}
    
    zero_config = {"stage": zero_stage}
    if zero_stage >= 2:
        zero_config.update({"allgather_bucket_size": 2e8, "reduce_bucket_size": 2e8,
                            "overlap_comm": True, "contiguous_gradients": True})
    if use_offload:
        zero_config["offload_optimizer"] = {"device": "cpu", "pin_memory": True}
        if zero_stage == 3:
            zero_config["offload_param"] = {"device": "cpu", "pin_memory": True}
    
    config["zero_optimization"] = zero_config
    
    print("Generated config:")
    print(json.dumps(config, indent=2))
    return config


# Test with different scenarios
print("SCENARIO 1: Small model (1.3B) on 4× A100-40GB")
c1 = deepspeed_config_wizard(1.3, gpu_memory_gb=40, num_gpus=4, gpu_generation='ampere')

print("\n" + "="*65)
print("SCENARIO 2: Medium model (7B) on 8× A100-80GB")
c2 = deepspeed_config_wizard(7, gpu_memory_gb=80, num_gpus=8, gpu_generation='ampere')

print("\n" + "="*65)
print("SCENARIO 3: Large model (70B) on 8× A100-80GB")
c3 = deepspeed_config_wizard(70, gpu_memory_gb=80, num_gpus=8, cpu_ram_gb=512, gpu_generation='ampere')

## Checkpointing and Inference with ZeRO

ZeRO-3 shards the model across GPUs, so saving/loading checkpoints requires special handling.

In [ ]:
# Checkpoint saving and loading patterns

print("Checkpoint saving with DeepSpeed:")
print("-" * 50)
print()
print("# SAVING (during training):")
print("  model_engine.save_checkpoint('./checkpoint_dir', tag='step_1000')")
print("  # Creates: checkpoint_dir/step_1000/")
print("  #           ├── mp_rank_00_model_states.pt  (model shard for GPU 0)")
print("  #           ├── mp_rank_01_model_states.pt  (model shard for GPU 1)")
print("  #           └── zero_pp_rank_0_mp_rank_00_optim_states.pt")
print()
print("# RESUMING training:")
print("  _, client_state = model_engine.load_checkpoint('./checkpoint_dir', tag='step_1000')")
print("  step = client_state['step']  # Restore custom state")
print()
print("# INFERENCE — consolidate ZeRO-3 shards to single file:")
print("  from deepspeed.utils.zero_to_fp32 import get_fp32_state_dict_from_zero_checkpoint")
print("  state_dict = get_fp32_state_dict_from_zero_checkpoint('./checkpoint_dir/step_1000')")
print("  model.load_state_dict(state_dict)  # Standard PyTorch model now")
print()
print("# Or use the CLI tool:")
print("  python -m deepspeed.utils.zero_to_fp32 ./checkpoint_dir/step_1000 model.pt")
print()
print("NOTE: With ZeRO-2, checkpoints are standard PyTorch format and torch.load() works.")

## Interview Questions and Answers

These are real questions asked in ML engineer interviews at top AI companies.

In [ ]:
interview_qa = [
    {
        "q": "What is ZeRO and what problem does it solve?",
        "a": """
ZeRO (Zero Redundancy Optimizer) solves the memory redundancy problem in distributed training.

In standard data-parallel training (DDP), every GPU keeps a full copy of:
  - Model weights (4 bytes/param in fp32)
  - Gradients (4 bytes/param)
  - Optimizer states — Adam stores momentum + variance (8 bytes/param)

Total: ~16 bytes/parameter. For a 7B model: 112 GB per GPU (!) — even 8× A100s can't fit this.

ZeRO partitions these tensors across GPUs, eliminating redundancy:
  - Stage 1: Optimizer states partitioned → ~4× memory saving
  - Stage 2: + Gradients partitioned     → ~8× memory saving
  - Stage 3: + Parameters partitioned    → ~N× (N = number of GPUs)

The trade-off is increased inter-GPU communication to gather shards when needed.
        """
    },
    {
        "q": "When would you choose DeepSpeed ZeRO over PyTorch FSDP?",
        "a": """
Choose DeepSpeed ZeRO when:
1. You need CPU/NVMe offload (ZeRO-Offload, ZeRO-Infinity) — FSDP barely supports this
2. Training very large models (100B+) on many nodes — DeepSpeed has better multi-node support
3. You're using Hugging Face + DeepSpeed workflow already established in your org
4. You need fine-grained control via JSON config (reproducibility, experiment tracking)

Choose PyTorch FSDP when:
1. You prefer pure PyTorch (no extra dependencies)
2. Simpler codebase maintenance is a priority
3. Your model fits on <16 GPUs without CPU offload
4. You're debugging and want easier introspection

Both achieve similar performance for ZeRO-3 equivalent workloads. FSDP is the long-term
direction for PyTorch native, but DeepSpeed remains more feature-rich today.
        """
    },
    {
        "q": "Explain gradient checkpointing and when you would use it.",
        "a": """
During the forward pass, PyTorch saves all intermediate activations for use in backward pass.
These activations grow linearly with model depth — a 100-layer model stores 100 activation tensors.

Gradient checkpointing saves memory by:
1. Only storing activations at checkpoint boundaries (e.g., every 10th layer)
2. Recomputing the intermediate activations during backward pass from the nearest checkpoint

Memory: reduces activation memory from O(L) to O(√L) — roughly √L recomputation segments.
Compute: ~33% more FLOPs since some forward passes run twice.

Use it when:
- Training large transformers where activations are the memory bottleneck
- You're batch-size limited by activation memory, not weights
- The 33% compute overhead is acceptable for your training timeline

In PyTorch: torch.utils.checkpoint.checkpoint(layer, input)
In HF Transformers: model.gradient_checkpointing_enable()
In DeepSpeed: enable in config or call model.enable_input_require_grads()
        """
    },
    {
        "q": "What is the difference between fp16 and bf16? When would you use each?",
        "a": """
Both are 16-bit floating point formats, but they allocate the 16 bits differently:

fp16 (IEEE 754 half-precision):
  - 1 sign bit, 5 exponent bits, 10 mantissa bits
  - Range: ~±65504 (very limited range!)
  - Risk: overflow when activations exceed 65504 → requires dynamic loss scaling
  - Supported: all CUDA GPUs (Volta+)

bf16 (Google Brain float):
  - 1 sign bit, 8 exponent bits, 7 mantissa bits
  - Same exponent as fp32 → same range: ~±3.4×10³⁸ (no overflow risk!)
  - Less precision than fp16, but range is identical to fp32
  - No loss scaling needed
  - Supported: Ampere+ (A100, RTX 3090), TPUs

Use fp16: on V100s, older GPUs, or when bf16 unavailable
Use bf16: whenever possible (A100+) — stabler training, no loss scaling complexity
        """
    },
    {
        "q": "What is ZeRO-Offload and what are its trade-offs?",
        "a": """
ZeRO-Offload extends ZeRO by moving tensors from GPU memory to CPU RAM:
  - Offload optimizer states to CPU: CPU holds momentum + variance tensors
  - Optimizer step runs on CPU — GPU handles only forward + backward pass
  - With ZeRO-3 + offload_param: also moves model weights to CPU between uses

Memory impact: A system with 8× 80GB GPUs and 512GB CPU RAM effectively has
  640 GB of 'GPU memory' — enough for training a 30B+ parameter model.

Trade-offs:
  Pros:
    - Train much larger models on existing hardware
    - CPU RAM is cheap ($2/GB) vs GPU memory ($25+/GB)
  Cons:
    - CPU optimizer step is 10-100× slower than GPU
    - PCIe bandwidth (32 GB/s) bottlenecks optimizer state transfers
    - GPU utilization drops — GPUs sit idle waiting for CPU optimizer
    - Ideal with large batch sizes to amortize optimizer overhead
        """
    },
    {
        "q": "How do you debug NaN losses in DeepSpeed fp16 training?",
        "a": """
NaN losses in fp16 training are almost always caused by gradient overflow (fp16 max = 65504).

Debugging steps:
1. Watch loss_scale in DeepSpeed logs — if it shrinks to 1 and stays there, you have overflow
   Look for: [deepspeed] OVERFLOW detected, loss_scale decreased to X

2. Check for NaN in specific layers:
   for name, param in model.named_parameters():
       if torch.isnan(param.grad).any():
           print(f'NaN gradient in {name}')

3. Solutions (in order of preference):
   a. Switch to bf16 if GPU supports it (eliminates overflow entirely)
   b. Reduce learning rate (smaller gradients = less overflow risk)
   c. Increase gradient clipping (clip_grad_norm)
   d. Increase loss_scale_window and hysteresis in fp16 config
   e. Add gradient checkpointing (may change gradient magnitudes slightly)

4. The fix DeepSpeed applies automatically: dynamic loss scaling
   - Multiplies loss by scale factor (e.g. 65536) before backward
   - Divides gradients by same factor after
   - Halves scale when overflow detected; grows when stable
        """
    },
]

for i, qa in enumerate(interview_qa, 1):
    print(f"Q{i}: {qa['q']}")
    print(f"A:  {qa['a'].strip()}")
    print("-" * 70)
    print()

## Summary and Next Steps

### What You Learned

| Concept | Key Takeaway |
|---------|-------------|
| Why training is memory-intensive | 16 bytes/param for weights + gradients + optimizer states |
| ZeRO Stage 1 | Shard optimizer states across GPUs (~4× savings) |
| ZeRO Stage 2 | + Gradient sharding (~8× savings) |
| ZeRO Stage 3 | + Weight sharding (~N× savings for N GPUs) |
| Mixed precision | fp16 (V100+, needs loss scaling) vs bf16 (A100+, stable) |
| Gradient checkpointing | Recompute activations to save memory (~33% compute cost) |
| ZeRO-Offload | Optimizer/params to CPU RAM — enables much larger models |
| ZeRO-Infinity | NVMe offload — terabyte-scale models |
| Config format | JSON with `train_batch_size`, `fp16`/`bf16`, `zero_optimization` |
| DeepSpeed vs FSDP | Both do model sharding; FSDP = simpler, DS = more features |

### Decision Flowchart

```
Model fits on 1 GPU? → Use standard PyTorch
        ↓
Fits across N GPUs? → Use ZeRO-1 or ZeRO-2
        ↓
Still too big? → Use ZeRO-3
        ↓
Still too big? → ZeRO-3 + CPU Offload (ZeRO-Offload)
        ↓
Still too big? → ZeRO-Infinity (NVMe)
```

### Next Steps

1. **Hugging Face DeepSpeed integration**: [https://huggingface.co/docs/transformers/deepspeed](https://huggingface.co/docs/transformers/deepspeed)
2. **Microsoft DeepSpeed blog posts**: [https://www.microsoft.com/en-us/research/blog/tag/deepspeed/](https://www.microsoft.com/en-us/research/blog/tag/deepspeed/)
3. **ZeRO original paper**: [https://arxiv.org/abs/1910.02054](https://arxiv.org/abs/1910.02054)
4. **Compare with PyTorch FSDP**: [https://pytorch.org/docs/stable/fsdp.html](https://pytorch.org/docs/stable/fsdp.html)
5. **DeepSpeed + LoRA (PEFT)**: Combine DeepSpeed for scale with LoRA for efficient fine-tuning
6. **Continue to MLOps**: Learn to track, version, and serve these large models in production